## Data Source (Education New Version)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## Data Source (Education New Version)

In [2]:
def load(path, dipl_col):
    cols = ["COMMUNE", "DCRAN", "AGEREVQ", "IPONDI", dipl_col]
    d = pd.read_csv(path, sep=";", dtype=str, usecols=cols)
    d["IPONDI"] = pd.to_numeric(d["IPONDI"].str.replace(",", "."))
    d = d.rename(columns={dipl_col: "DIPL"})
    return d[d["AGEREVQ"].astype(int) >= 25]

d13 = load("FD_MIGCOM_2013.txt", "DIPL_15")
d18 = load("FD_MIGCOM_2018.csv", "DIPL")
d23 = load("FD_MIGCOM_2023.csv", "DIPL")

In [3]:
def to_commune(s):
    # Clean the codes: text, no stray spaces, always 5 characters (1001 -> 01001)
    s = s.astype(str).str.strip().str.zfill(5)

    # mask(condition, new_value) replaces values where the condition is true
    # and leaves everything else unchanged

    # Paris: 75101 to 75120 -> 75056
    s = s.mask(s.str.match(r"^751(0[1-9]|1\d|20)$"), "75056")

    # Lyon: 69381 to 69389 -> 69123
    s = s.mask(s.str.match(r"^6938[1-9]$"), "69123")

    # Marseille: 13201 to 13216 -> 13055
    s = s.mask(s.str.match(r"^132(0[1-9]|1[0-6])$"), "13055")

    return s

# Apply to current and previous commune in all three files
d13["COMMUNE"], d13["DCRAN"] = to_commune(d13["COMMUNE"]), to_commune(d13["DCRAN"])
d18["COMMUNE"], d18["DCRAN"] = to_commune(d18["COMMUNE"]), to_commune(d18["DCRAN"])
d23["COMMUNE"], d23["DCRAN"] = to_commune(d23["COMMUNE"]), to_commune(d23["DCRAN"])

In [4]:
# City Perpignan - check on in an outflow
c = "66136"

# Population of Perpignan and in/outflow for 2013, 2018, 2023
for name, d in {"2013": d13, "2018": d18, "2023": d23}.items():
    in_c = d[(d["COMMUNE"] == c) & (d["DCRAN"] != c) & (d["DCRAN"] != "99999")]
    out_c = d[(d["DCRAN"] == c) & (d["COMMUNE"] != c)]
    now = d[d["COMMUNE"] == c]
    print(f"{name}: residents {now['IPONDI'].sum():,.0f} | "
          f"in {in_c['IPONDI'].sum():,.0f} | out {out_c['IPONDI'].sum():,.0f}")

2013: residents 82,551 | in 4,229 | out 4,662
2018: residents 82,683 | in 4,045 | out 4,747
2023: residents 84,720 | in 3,978 | out 4,258


#### Variables in the INSEE migration files (RP2013, RP2018, RP2023)

| Variable | Meaning |
|---|---|
| `COMMUNE` | Commune where the person lives at the census |
| `DCRAN` | Commune where the person lived on 1 January of the previous year (`99999` = abroad) |
| `AGEREVQ` | Age in 5-year groups; only ages 25+ are kept |
| `DIPL` | Highest diploma (`DIPL_15` in RP2013, renamed here) |
| `IPONDI` | Survey weight: how many people one row represents |

**Moves for a commune c:**
- **Moved in:** `COMMUNE` = c and `DCRAN` ≠ c, excluding `DCRAN` = 99999
- **Moved out:** `DCRAN` = c and `COMMUNE` ≠ c
- **Residents:** `COMMUNE` = c

Counts are weighted, so they sum `IPONDI` rather than counting rows. Moves abroad are not observed, so in and out flows cover moves within France only.

## Define new several outcome variables

In [5]:
# Define education levels (according to variable definitions of 2013, 2018, 2023) PDFs
EDU = {
    "d13": {"bac_plus": ["C", "D"], "tertiary": ["D"],
            "bac2": [], "bac34": [], "bac5_plus": []},  # not available in RP2013
    "d18": {"bac_plus": ["14", "15", "16", "17", "18", "19"], "tertiary": ["16", "17", "18", "19"],
            "bac2": ["16"], "bac34": ["17"], "bac5_plus": ["18", "19"]},
    "d23": {"bac_plus": ["105", "106", "108", "109", "110", "111"], "tertiary": ["108", "109", "110", "111"],
            "bac2": ["108"], "bac34": ["109"], "bac5_plus": ["110", "111"]},
}

In [6]:
"""
 Define education levels (according to variable definitions of 2013, 2018, 2023)
EDU = {
    "d13": {"bac_plus": ["C", "D"], "tertiary": ["D"]},
    "d18": {"bac_plus": ["14", "15", "16", "17", "18", "19"], "tertiary": ["16", "17", "18", "19"]},
    "d23": {"bac_plus": ["105", "106", "108", "109", "110", "111"], "tertiary": ["108", "109", "110", "111"]},
}
"""

'\n Define education levels (according to variable definitions of 2013, 2018, 2023)\nEDU = {\n    "d13": {"bac_plus": ["C", "D"], "tertiary": ["D"]},\n    "d18": {"bac_plus": ["14", "15", "16", "17", "18", "19"], "tertiary": ["16", "17", "18", "19"]},\n    "d23": {"bac_plus": ["105", "106", "108", "109", "110", "111"], "tertiary": ["108", "109", "110", "111"]},\n}\n'

In [7]:
# Helper function to add education level columns to the dataframes
def add_edu(d, codes):
    d = d.copy()
    d["bac_plus"] = d["DIPL"].isin(codes["bac_plus"])
    d["tertiary"] = d["DIPL"].isin(codes["tertiary"])
    d["no_bac"] = ~d["bac_plus"]

    # Detailed tertiary levels (RP2018 and RP2023 only)
    for g in ["bac2", "bac34", "bac5_plus"]:
        d[g] = d["DIPL"].isin(codes[g])

    return d

# Add education level columns to the dataframes
d13 = add_edu(d13, EDU["d13"])
d18 = add_edu(d18, EDU["d18"])
d23 = add_edu(d23, EDU["d23"])

# Loop through the dataframes and print the weighted average of education levels for each year
for name, d in {"2013": d13, "2018": d18, "2023": d23}.items():
    w = d["IPONDI"]
    print(name, sorted(d["DIPL"].unique()),
          {g: round((w * d[g]).sum() / w.sum(), 3) for g in ["no_bac", "bac_plus", "tertiary"]})

2013 ['A', 'B', 'C', 'D'] {'no_bac': 0.564, 'bac_plus': 0.436, 'tertiary': 0.277}
2018 ['01', '02', '03', '11', '12', '13', '14', '15', '16', '17', '18', '19'] {'no_bac': 0.523, 'bac_plus': 0.477, 'tertiary': 0.315}
2023 ['101', '102', '103', '104', '105', '106', '108', '109', '110', '111'] {'no_bac': 0.475, 'bac_plus': 0.525, 'tertiary': 0.354}


In [8]:
# Diploma groups to calculate ("all" = every adult aged 25+)
GROUPS = ["all", "no_bac", "bac_plus", "tertiary", "bac2", "bac34", "bac5_plus"]


# GROUPS = ["all", "no_bac", "bac_plus", "tertiary"]

In [9]:
def collapse(d, vintage):
    # Add a helper column so "all" works like the other groups
    d = d.assign(all=True)
    tables = []

    for g in GROUPS:
        # Keep only people in this diploma group
        people = d[d[g]]

        # True if current and previous commune differ
        moved = people["COMMUNE"] != people["DCRAN"]
        # True if the person lived abroad one year earlier
        abroad = people["DCRAN"] == "99999"

        # Weighted residents, counted at their current commune
        residents = people.groupby("COMMUNE")["IPONDI"].sum()
        # Moved in from elsewhere in France, counted at the new commune
        inflow = people[moved & ~abroad].groupby("COMMUNE")["IPONDI"].sum()
        # Moved in from abroad, kept separately
        from_abroad = people[abroad].groupby("COMMUNE")["IPONDI"].sum()
        # Moved out, counted at the previous commune (note: grouped by DCRAN)
        outflow = people[moved & ~abroad].groupby("DCRAN")["IPONDI"].sum()

        # Line the four counts up by commune; no moves means 0, not missing
        t = pd.DataFrame({
            "residents": residents,
            "inflow": inflow,
            "from_abroad": from_abroad,
            "outflow": outflow,
        }).fillna(0)

        # Turn the commune index into a column and label the row
        t = t.rename_axis("code_commune").reset_index()
        t["group"] = g
        t["vintage"] = vintage
        tables.append(t)

    # One table with all groups for this file
    return pd.concat(tables, ignore_index=True)

# Collapse each file and stack them
flows = pd.concat(
    [collapse(d13, 2013), collapse(d18, 2018), collapse(d23, 2023)],
    ignore_index=True,
)

# Check 1: one row per commune x group x vintage (should print 0)
print("Duplicates:", flows.duplicated(["code_commune", "group", "vintage"]).sum())

# Check 2: Perpignan, to compare with the earlier hand calculation
print(flows[(flows["code_commune"] == "66136") & (flows["group"] == "all")])
print(flows.shape)

Duplicates: 0
       code_commune     residents       inflow  from_abroad      outflow  \
27378         66136  82550.817849  4229.213695   361.952640  4661.961777   
171873        66136  82682.815585  4045.003962   385.670574  4746.972282   
404931        66136  84719.723186  3978.028432   382.470383  4257.932977   

       group  vintage  
27378    all     2013  
171873   all     2018  
404931   all     2023  
(613783, 7)


In [10]:
flows.sample(3) #nice

,code_commune,residents,inflow,from_abroad,outflow,group,vintage
549743,88115,143.286175,4.871190,4.87119,0.000000,bac2,2023
550604,91067,10.571429,0.000000,0.00000,0.000000,bac2,2023
452652,11393,203.303620,14.720114,0.00000,4.762411,bac_plus,2023


In [11]:
# Rebuild residents one year earlier:
# take today's residents, remove everyone who arrived, add back everyone who left
flows["base"] = (
    flows["residents"] - flows["inflow"] - flows["from_abroad"] + flows["outflow"]
)

# Less than one (weighted) earlier resident: rate undefined, so set the base to missing.
# This also catches tiny positive leftovers from floating-point rounding.
flows["base"] = flows["base"].where(flows["base"] >= 1)

# Yearly move-in rate: arrivals from elsewhere in France / earlier residents
flows["in_rate"] = flows["inflow"] / flows["base"]

# Yearly move-out rate: departures within France / earlier residents
flows["out_rate"] = flows["outflow"] / flows["base"]

# Net rate: positive means the commune gained people in this group
flows["net_rate"] = flows["in_rate"] - flows["out_rate"]

# Check Perpignan in all groups and vintages
cols = ["vintage", "group", "base", "in_rate", "out_rate", "net_rate"]
print(flows.loc[flows["code_commune"] == "66136", cols].round(4).to_string())

# Check the distribution in all communes
print(flows.groupby(["vintage", "group"])["net_rate"].describe().round(3))

        vintage      group        base  in_rate  out_rate  net_rate
27378      2013        all  82621.6133   0.0512    0.0564   -0.0052
64040      2013     no_bac  47711.0406   0.0409    0.0444   -0.0035
100594     2013   bac_plus  34910.5727   0.0652    0.0729   -0.0076
136693     2013   tertiary  21175.9140   0.0665    0.0767   -0.0102
171873     2018        all  82999.1133   0.0487    0.0572   -0.0085
206826     2018     no_bac  44870.7819   0.0395    0.0467   -0.0072
241703     2018   bac_plus  38128.3315   0.0596    0.0695   -0.0099
276218     2018   tertiary  23524.6105   0.0651    0.0732   -0.0081
309513     2018       bac2   7859.0981   0.0640    0.0780   -0.0141
341353     2018      bac34   7796.6750   0.0653    0.0719   -0.0066
371194     2018  bac5_plus   7868.8374   0.0662    0.0697   -0.0036
404931     2023        all  84617.1573   0.0470    0.0503   -0.0033
439773     2023     no_bac  43184.7996   0.0354    0.0382   -0.0028
474584     2023   bac_plus  41432.3577   0.0591 

### Load existing df

In [12]:
# Load the election data, keeping commune codes as text
df_rdd = pd.read_csv("df_rdd_clean.csv", dtype={"code_commune": str})

# Restore the 5-character format, e.g. 1001 -> 01001
df_rdd["code_commune"] = df_rdd["code_commune"].str.zfill(5)

In [13]:
# Setup definitions

# Which census vintage is "pre" and "post" for each election
CYCLES = {
    2014: {"pre": 2013, "post": 2018},
    2020: {"pre": 2018, "post": 2023},
}

# Columns to carry over, and the diploma groups
MEASURES = ["in_rate", "out_rate", "net_rate", "base"]
# GROUPS = ["all", "no_bac", "bac_plus", "tertiary"]
GROUPS = ["all", "no_bac", "bac_plus", "tertiary", "bac2", "bac34", "bac5_plus"]

### Build pre- and post tables

In [14]:
def to_wide(vintage, period):
    # One vintage: from one row per commune x group to one row per commune
    wide = flows[flows["vintage"] == vintage].pivot(
        index="code_commune", columns="group", values=MEASURES
    )
    # Name the columns, e.g. net_rate_tertiary_pre
    wide.columns = [f"{measure}_{group}_{period}" for measure, group in wide.columns]
    return wide


cycle_tables = []
for election, v in CYCLES.items():
    # Put pre and post side by side for each commune in this election
    t = to_wide(v["pre"], "pre").join(to_wide(v["post"], "post"), how="outer")
    t = t.reset_index()
    # Label the election, which is the second merge key
    t["election_year"] = election
    cycle_tables.append(t)

# One table with one row per commune x election
outcomes = pd.concat(cycle_tables, ignore_index=True)
print(outcomes.shape)

(71716, 58)


In [15]:
# Change from pre to post, e.g. delta_net_rate_tertiary (main outcome)
for group in GROUPS:
    for measure in ["in_rate", "out_rate", "net_rate"]:
        outcomes[f"delta_{measure}_{group}"] = (
            outcomes[f"{measure}_{group}_post"] - outcomes[f"{measure}_{group}_pre"]
        )

### Merge with df_rdd (left merge, so drop all the other communes (not in treated/ control group))

In [16]:
# Left merge: keeps exactly the rows of df_rdd and ignores all other communes
n_before = len(df_rdd)
df_final = df_rdd.merge(
    outcomes, on=["code_commune", "election_year"], how="left", validate="1:1"
)

# Stop here if the merge added or lost rows
assert len(df_final) == n_before
print(df_final.shape)

(837, 102)


In [17]:
# Only 2014 and 2020 get the new outcomes; 2008 stays empty
sample = df_final[df_final["election_year"].isin(CYCLES)]

# How many rows and treated communes got an outcome
print(sample.groupby("election_year").agg(
    rows=("treat", "size"),
    treated=("treat", "sum"),
    matched=("delta_net_rate_tertiary", "count"),
))

# Communes without an outcome
print(sample.loc[
    sample["delta_net_rate_tertiary"].isna(),
    ["code_commune", "election_year", "treat"],
])

# The range of the main outcome inside your sample
print(sample[[
    "net_rate_tertiary_pre", "net_rate_tertiary_post", "delta_net_rate_tertiary",
]].describe().round(4))

               rows  treated  matched
election_year                        
2014            513       13      513
2020            285       13      285
Empty DataFrame
Columns: [code_commune, election_year, treat]
Index: []
       net_rate_tertiary_pre  net_rate_tertiary_post  delta_net_rate_tertiary
count               798.0000                798.0000                 798.0000
mean                 -0.0052                 -0.0032                   0.0020
std                   0.0300                  0.0285                   0.0363
min                  -0.1373                 -0.1183                  -0.1754
25%                  -0.0220                 -0.0195                  -0.0161
50%                  -0.0071                 -0.0050                   0.0016
75%                   0.0102                  0.0126                   0.0186
max                   0.1462                  0.1489                   0.2545


### Drop old outcome var and export

In [18]:
# Remove the old outcome; errors="ignore" keeps this safe if a column is already gone
OLD_OUTCOMES = ["delta_skill_share", "skill_share_trend2"]
df_final = df_final.drop(columns=OLD_OUTCOMES, errors="ignore")

In [19]:
# Save as a new file (leaving df_rdd_clean untouched)
df_final.to_csv("df_diffindisc_clean.csv", index=False)